# 00 — Entorno de ejecución

**Objetivo de este notebook:** comprobar, con evidencia y sin suposiciones, en qué máquina va a correr el entrenamiento de SkillGen (QLoRA sobre Qwen2.5-7B-Instruct).

La regla del proyecto es *honestidad primero*: si algo no está (por ejemplo, `torch` o una GPU en esta VM), se reporta tal cual. Este notebook **nunca falla por no tener GPU**: termina bien e imprime un resumen `WARN` claro. El entrenamiento real está pensado para **Kaggle** (GPU T4 gratuita), y este mismo notebook, ejecutado allí, debe dar el veredicto «Kaggle-ready».

Qué vamos a verificar, paso a paso:

1. Versión de Python y plataforma.
2. Si `torch` está instalado, si hay CUDA y qué GPU es (nombre, *compute capability* y VRAM).
3. Espacio en disco disponible.
4. Versiones de las librerías clave.
5. Si estamos dentro de Kaggle.
6. La **regla de reserva** (*fallback*) de longitud de secuencia según la VRAM detectada.

## 1. Python y plataforma

Empezamos por lo básico: el intérprete exacto y el sistema. SkillGen exige **Python ≥ 3.12** (ver `pyproject.toml`).

In [1]:
import platform
import sys

print("Python:", sys.version)
print("Ejecutable:", sys.executable)
print("Plataforma:", platform.platform())
print("Máquina:", platform.machine())
assert sys.version_info >= (3, 12), "SkillGen requiere Python >= 3.12"
print("OK: Python >= 3.12")

Python: 3.12.3 (main, Aug 31 2026, 10:18:26) [GCC 13.3.0]
Ejecutable: /home/hatch/workspace/skillgen/.venv/bin/python
Plataforma: Linux-7.0.0-39-generic-x86_64-with-glibc2.39
Máquina: x86_64
OK: Python >= 3.12


## 2. GPU y CUDA (con honestidad si no hay)

`torch` **no** es dependencia de este repositorio en la VM local: aquí solo usamos el *tokenizer* (CPU). Por eso el `import torch` va dentro de un `try/except`: si no está, lo decimos y seguimos — en Kaggle sí estará, porque el entorno de Kaggle lo trae preinstalado para el entrenamiento.

Si hay GPU, necesitamos dos datos para decidir la configuración de entrenamiento:

- **Compute capability ≥ 7.0**: es el mínimo razonable para QLoRA con cuantización de 4 bits en una T4 (la T4 es 7.5).
- **VRAM**: determina la longitud de secuencia segura (ver la regla de reserva al final).

In [2]:
GPU_INFO = {"present": False, "name": None, "compute_capability": None, "vram_gb": None}

try:
    import torch

    print("torch:", torch.__version__)
    print("CUDA disponible:", torch.cuda.is_available())
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        cc = (props.major, props.minor)
        GPU_INFO.update(
            present=True,
            name=props.name,
            compute_capability=float(f"{props.major}.{props.minor}"),
            vram_gb=props.total_memory / (1024**3),
        )
        print("GPU:", props.name)
        print("Compute capability:", f"{cc[0]}.{cc[1]}")
        print("VRAM total (GB):", round(GPU_INFO["vram_gb"], 2))
    else:
        print("WARN: torch está instalado pero CUDA NO está disponible (sin GPU).")
except ImportError:
    print("WARN: torch NO está instalado en este entorno.")
    print("Es lo esperado en la VM local de CPU: aquí solo corre el tokenizer.")
    print("El entrenamiento se ejecutará en Kaggle, donde torch + GPU vienen de serie.")

print()
print("GPU_INFO =", GPU_INFO)

WARN: torch NO está instalado en este entorno.
Es lo esperado en la VM local de CPU: aquí solo corre el tokenizer.
El entrenamiento se ejecutará en Kaggle, donde torch + GPU vienen de serie.

GPU_INFO = {'present': False, 'name': None, 'compute_capability': None, 'vram_gb': None}


## 3. Espacio en disco

El modelo base (7B) más los adaptadores y checkpoints ocupan varios GB. Comprobamos cuánto disco libre hay en el directorio del proyecto.

In [3]:
import shutil
from pathlib import Path

total, used, free = shutil.disk_usage(Path.cwd())
print(f"Disco total: {total / 1024**3:.1f} GB")
print(f"Disco usado: {used / 1024**3:.1f} GB")
print(f"Disco libre: {free / 1024**3:.1f} GB")

Disco total: 100.0 GB
Disco usado: 29.5 GB
Disco libre: 69.7 GB


## 4. Versiones de librerías clave

Registramos las versiones exactas para que cualquier resultado sea reproducible. Si una librería falta, se imprime `no instalada` en lugar de romper el notebook.

In [4]:
import importlib.metadata as md

for package in ["transformers", "tokenizers", "matplotlib", "numpy", "pytest", "torch", "peft", "trl", "bitsandbytes"]:
    try:
        print(f"{package}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package}: no instalada")

transformers: 5.20.2
tokenizers: 0.23.3
matplotlib: 3.11.2
numpy: 2.5.4
pytest: 9.1.1
torch: no instalada
peft: no instalada
trl: no instalada
bitsandbytes: no instalada


## 5. ¿Estamos en Kaggle?

Kaggle se detecta por la variable de entorno `KAGGLE_KERNEL_RUN_TYPE` o por la existencia de la ruta `/kaggle`. Esto importa porque la configuración de entrenamiento (y dónde se guardan los resultados) depende de dónde corremos.

In [5]:
import os
from pathlib import Path

on_kaggle = ("KAGGLE_KERNEL_RUN_TYPE" in os.environ) or Path("/kaggle").exists()
print("KAGGLE_KERNEL_RUN_TYPE:", os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
print("¿Estamos en Kaggle?", on_kaggle)

KAGGLE_KERNEL_RUN_TYPE: None
¿Estamos en Kaggle? False


## 6. Veredicto y regla de reserva de longitud de secuencia

**Regla de reserva (fallback) documentada del proyecto:**

> Si la VRAM es **< 16 GB** o la compute capability es **< 7.0** → `max_seq_length = 2048`, `per_device_batch = 1` y se compensa con **más acumulación de gradiente** (*gradient accumulation*) para mantener el tamaño efectivo de lote.

La decisión principal (medida en el EDA de la Fase 0, notebook `01_dataset_eda.ipynb`) es `max_seq_length = 8192` para una T4 de 16 GB. Esta regla de reserva protege el entrenamiento en GPUs más pequeñas: bajar la secuencia reduce la memoria de activaciones, y la acumulación de gradiente recupera el lote efectivo sin necesitar más VRAM.

En una máquina **sin GPU** (como esta VM) el veredicto honesto es `WARN`: no se puede entrenar aquí, y el notebook termina bien igualmente. En una máquina **con GPU** exigimos compute capability ≥ 7.0 con un `assert`.

In [6]:
def recommended_config(gpu: dict) -> dict:
    """Configuración recomendada según la GPU detectada (regla de reserva)."""
    if not gpu["present"]:
        # Sin GPU no hay entrenamiento posible: devolvemos la config de reserva
        # como referencia para la máquina más pequeña que sí podría entrenar.
        return {"max_seq_length": 2048, "per_device_batch": 1,
                "gradient_accumulation_steps": 16, "nota": "fallback (sin GPU detectada)"}
    if gpu["vram_gb"] < 16 or gpu["compute_capability"] < 7.0:
        return {"max_seq_length": 2048, "per_device_batch": 1,
                "gradient_accumulation_steps": 16, "nota": "fallback (VRAM < 16 GB o CC < 7.0)"}
    return {"max_seq_length": 8192, "per_device_batch": 1,
            "gradient_accumulation_steps": 16, "nota": "principal (T4 16 GB o mejor)"}


config = recommended_config(GPU_INFO)
print("Configuración recomendada:", config)
print()
if GPU_INFO["present"]:
    assert GPU_INFO["compute_capability"] >= 7.0, "Se requiere compute capability >= 7.0"
    print("VEREDICTO: Kaggle-ready — GPU con compute capability >= 7.0 detectada.")
else:
    print("VEREDICTO: WARN — entorno SIN GPU (solo CPU).")
    print("Este notebook termina OK a propósito: su trabajo aquí es auditar, no entrenar.")
    print("El entrenamiento QLoRA se ejecutará en Kaggle con este mismo notebook como puerta de entrada.")

Configuración recomendada: {'max_seq_length': 2048, 'per_device_batch': 1, 'gradient_accumulation_steps': 16, 'nota': 'fallback (sin GPU detectada)'}

VEREDICTO: WARN — entorno SIN GPU (solo CPU).
Este notebook termina OK a propósito: su trabajo aquí es auditar, no entrenar.
El entrenamiento QLoRA se ejecutará en Kaggle con este mismo notebook como puerta de entrada.
